# BCGGAN: aplicacion de un modelo entrenado

Este notebook carga solo EEG contaminado y un checkpoint BCGGAN previamente entrenado. Aplica el generador BCG-a-EEG limpio y muestra una comparacion temporal. No calcula metricas: quedan pendientes de definir en el proyecto.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import mne
import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from functions.bcggan import BCGGANTrainer

In [ ]:
EEG_ROOT = PROJECT_ROOT / 'data/raw/Dataset1/Simultaneous_EEG_fMRI/BIDS_dataset_EEG'
SUBJECT = 'sub-001'
TASK = 'fmrirestingec'
CHECKPOINT = PROJECT_ROOT / 'data/models/bcggan_sub-001.pt'
CHANNEL_INDEX = 0
START_SECONDS = 0.0
DISPLAY_SECONDS = 10.0
WINDOW_SECONDS = 1.0
BATCH_SIZE = 32

In [ ]:
eeg_path = EEG_ROOT / SUBJECT / 'eeg' / f'{SUBJECT}_task-{TASK}_eeg.set'
if not eeg_path.is_file():
    raise FileNotFoundError(f'EEGLAB header not found: {eeg_path}')
if not CHECKPOINT.is_file():
    raise FileNotFoundError(f'BCGGAN checkpoint not found: {CHECKPOINT}')

raw = mne.io.read_raw_eeglab(eeg_path, preload=True, verbose='ERROR')
corrupted = raw.get_data()
fs = float(raw.info['sfreq'])
trainer = BCGGANTrainer.load_checkpoint(CHECKPOINT)
print(f'EEG: {eeg_path.name}; shape: {corrupted.shape}; fs: {fs} Hz')
print(f'Checkpoint: {CHECKPOINT.name}; completed epochs: {trainer.completed_epochs}')

In [ ]:
mne.viz.set_browser_backend('qt')
raw.plot(
    n_channels=20,
    duration=10,
    show_scrollbars=True,
    block=True,
)

In [ ]:
if trainer.config.channels == 1:
    auxiliary_tokens = ('ECG', 'EKG', 'VREF', 'TRIG', 'STI', 'MISC', 'RESP', 'EOG', 'EMG', 'AUX')
    eeg_indices = [index for index, name in enumerate(raw.ch_names) if not any(token in name.upper() for token in auxiliary_tokens)]
    cleaned = corrupted.copy()
    cleaned[eeg_indices] = trainer.clean_recording(corrupted[eeg_indices], fs, window_s=WINDOW_SECONDS, stride_s=WINDOW_SECONDS, batch_size=BATCH_SIZE)
else:
    cleaned = trainer.clean_recording(corrupted, fs, window_s=WINDOW_SECONDS, stride_s=WINDOW_SECONDS, batch_size=BATCH_SIZE)
# TODO: incorporar metricas de evaluacion del conjunto de test cuando esten definidas.

## Visualizacion despues de BCGGAN

El resultado se inserta en una copia de `Raw` para utilizar el mismo visor interactivo de MNE que AAS.

In [ ]:
raw_clean = raw.copy()
raw_clean._data = cleaned.copy()
raw_clean.plot(
    n_channels=20,
    duration=10,
    show_scrollbars=True,
    block=True,
)

In [ ]:
start = int(round(START_SECONDS * fs))
stop = min(start + int(round(DISPLAY_SECONDS * fs)), corrupted.shape[1])
time = np.arange(start, stop) / fs
channel_name = raw.ch_names[CHANNEL_INDEX]

fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True)
axes[0].plot(time, corrupted[CHANNEL_INDEX, start:stop], color='tab:blue', lw=0.8)
axes[0].set_title(f'EEG contaminado | {channel_name}')
axes[1].plot(time, cleaned[CHANNEL_INDEX, start:stop], color='tab:orange', lw=0.8)
axes[1].set_title(f'EEG limpiado por BCGGAN | {channel_name}')
for axis in axes:
    axis.set_ylabel('Amplitud (V)')
    axis.grid(alpha=0.3)
axes[-1].set_xlabel('Tiempo (s)')
plt.tight_layout(); plt.show()

plt.figure(figsize=(15, 4))
plt.plot(time, corrupted[CHANNEL_INDEX, start:stop], label='EEG contaminado', alpha=0.7, lw=0.8)
plt.plot(time, cleaned[CHANNEL_INDEX, start:stop], label='BCGGAN', alpha=0.8, lw=0.8)
plt.title(f'Antes/despues de BCGGAN | {channel_name}')
plt.xlabel('Tiempo (s)'); plt.ylabel('Amplitud (V)')
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()